# Comparação: com vs sem remoção de outliers

O `rm_outliers` por IQR estatístico remove ~27% do dataset — mas justamente a região do espaço onde a Terra vive (planetas com período de ~ 1 ano são raros por viés observacional). Este notebook roda o pipeline nas duas configurações e compara os efeitos.

**Requisitos**: rode antes
```bash
python run_pipeline.py
python run_pipeline.py --no-outliers --out outputs_no_outliers
```

> 💡 **Sugestão:** o notebook `comparacao_metodos_outliers.ipynb` estende esta análise para **quatro métodos** (IQR, Z-score, MAD, none). Este notebook mostra apenas a comparação binária IQR vs sem filtro.

> **Nota sobre execução:** este notebook detecta automaticamente se está rodando no Google Colab ou localmente. No Colab, monta seu Google Drive e assume que a pasta `exoplanetas/` está em `/content/drive/MyDrive/exoplanetas`. Ajuste o caminho na célula abaixo se necessário.

In [ ]:
# ─── Setup: detecta Colab e configura ambiente ────────────────
# Esta célula funciona tanto em Jupyter local quanto no Google Colab.
import sys
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # >>> AJUSTE o caminho do projeto no seu Drive se necessário <<<
    PROJECT_ROOT = Path('/content/drive/MyDrive/exoplanetas')
    # instala dependências que o Colab não tem por padrão
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'openpyxl'], check=False)
else:
    # Local: assume que este notebook está em <projeto>/notebooks/
    PROJECT_ROOT = Path.cwd().parent

assert PROJECT_ROOT.exists(), f'Projeto não encontrado em: {PROJECT_ROOT}'
sys.path.insert(0, str(PROJECT_ROOT))

print(f'Ambiente: {"Google Colab" if IN_COLAB else "Local"}')
print(f'Projeto:  {PROJECT_ROOT}')

from pipeline import preprocessing, diagnostics
DATA = PROJECT_ROOT / "data" / "PSCompData.xlsx"
OUT_COM = PROJECT_ROOT / "outputs"
OUT_SEM = PROJECT_ROOT / "outputs_none"

In [ ]:
# Setup específico deste notebook
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

OUT_COM = PROJECT_ROOT / "outputs"          # execução padrão (IQR)
OUT_SEM = PROJECT_ROOT / "outputs_none"     # execução sem filtro de outliers

## 1. Contagem de amostras e distribuição de rótulos

In [ ]:
df_com = pd.read_excel(OUT_COM / "DFsvm.xlsx", index_col=0)
df_sem = pd.read_excel(OUT_SEM / "DFsvm.xlsx", index_col=0)

print(f"{'':30} {'COM outliers':>15} {'SEM outliers':>15}")
print(f"{'Total de amostras':30} {len(df_com):>15,} {len(df_sem):>15,}")
print()
print("Tamanho dos clusters:")
cc = df_com['cluster_hc'].value_counts().sort_index()
cs = df_sem['cluster_hc'].value_counts().sort_index()
for c in sorted(set(cc.index) | set(cs.index)):
    print(f"  Cluster {c:2}                    {cc.get(c, 0):>15,} {cs.get(c, 0):>15,}")
print()
print("Distribuição de rótulos (LP):")
for lab in sorted(set(df_com['label_lp'].unique()) | set(df_sem['label_lp'].unique())):
    n_com = (df_com['label_lp'] == lab).sum()
    n_sem = (df_sem['label_lp'] == lab).sum()
    nome = 'TT (Terrestre)' if lab == 1 else 'NT (Não-terrestre)'
    print(f"  {nome:30} {n_com:>15,} {n_sem:>15,}")

**Leitura esperada**: sem remoção de outliers, o clustering fica dominado por poucos objetos extremos (brown dwarfs em órbita larga) — um cluster com pouquíssimos elementos, e o resto colapsa numa única categoria.

## 2. Quem são os "outliers" removidos pelo IQR?

Diferença entre os dois datasets → planetas que o filtro estatístico removeu.

In [ ]:
nomes_com = set(df_com['pl_name'])
nomes_sem = set(df_sem['pl_name'])
removidos = nomes_sem - nomes_com

print(f"O filtro IQR removeu {len(removidos)} planetas ({100*len(removidos)/len(df_sem):.1f}% do dataset).")

# Ver características dos removidos — carrega dataset bruto pra pegar valores físicos
df_raw = preprocessing.carregar_arquivo(str(DATA))
df_removidos = df_raw[df_raw['pl_name'].isin(removidos)][
    ['pl_name', 'pl_orbper', 'pl_orbsmax', 'pl_rade', 'pl_bmasse', 'pl_orbeccen']
].dropna()

print("\nEstatísticas dos planetas REMOVIDOS pelo IQR (valores físicos originais):")
print(df_removidos.describe()[['pl_orbper', 'pl_rade', 'pl_bmasse']].round(2))

print("\nEstatísticas dos planetas MANTIDOS:")
df_mantidos = df_raw[df_raw['pl_name'].isin(nomes_com)][
    ['pl_name', 'pl_orbper', 'pl_orbsmax', 'pl_rade', 'pl_bmasse', 'pl_orbeccen']
].dropna()
print(df_mantidos.describe()[['pl_orbper', 'pl_rade', 'pl_bmasse']].round(2))

# Terra tem pl_orbper=365, pl_rade=1, pl_bmasse=1 — vejamos onde ela cai:
print("\nReferência: Terra tem orbper=365.25 dias, rade=1.0, bmasse=1.0")
print(f"→ Mediana dos MANTIDOS: orbper={df_mantidos['pl_orbper'].median():.1f}, "
      f"rade={df_mantidos['pl_rade'].median():.2f}, "
      f"bmasse={df_mantidos['pl_bmasse'].median():.1f}")

## 3. Validação externa nos dois cenários

Rodo `diagnostics.validacao_externa` reajustando o scaler em cada versão do dataset — para ver quão longe a Terra fica em z-score em cada configuração.

In [ ]:
# Preciso do scaler de cada pipeline. A forma mais simples: re-executar só a etapa 1.
# (não plotamos, só queremos os objetos)

res_com = preprocessing.main(
    caminho_entrada=str(DATA),
    caminho_saida=str(OUT_COM / "_tmp_com.xlsx"),
    caminho_rgjson=str(OUT_COM / "_tmp_ranges.json"),
    remover_outliers=True, save_dir=None, show=False,
)

res_sem = preprocessing.main(
    caminho_entrada=str(DATA),
    caminho_saida=str(OUT_SEM / "_tmp_sem.xlsx"),
    caminho_rgjson=str(OUT_SEM / "_tmp_ranges.json"),
    outlier_method='none', save_dir=None, show=False,
)

# Padroniza cada corpo de referência com os dois scalers
corpos = diagnostics.CORPOS_REFERENCIA
X_com = res_com['scaler'].transform(corpos[diagnostics.COLUNAS_NUMERICAS])
X_sem = res_sem['scaler'].transform(corpos[diagnostics.COLUNAS_NUMERICAS])

tabela = pd.DataFrame({
    'nome': corpos['nome'],
    'esperado': corpos['tipo_esperado'],
    '|z|_max (COM)': np.abs(X_com).max(axis=1).round(2),
    '|z|_max (SEM)': np.abs(X_sem).max(axis=1).round(2),
})
tabela['fator_reducao'] = (tabela['|z|_max (COM)'] / tabela['|z|_max (SEM)']).round(1)
print(tabela.to_string(index=False))

**Leitura**: quanto menor `|z|_max`, mais próximo o corpo está da distribuição de treino. Se o `|z|_max` da Terra cair drasticamente ao remover o filtro IQR, é sinal de que o filtro tava jogando a Terra pra fora.

## 4. Similaridade com a Terra

Top 10 planetas mais similares à Terra em cada cenário.

In [ ]:
sim_com = pd.read_csv(OUT_COM / "similaridade_exoplanetas_terra.csv")
sim_sem = pd.read_csv(OUT_SEM / "similaridade_exoplanetas_terra.csv")

print(f"COM outliers removidos ({len(sim_com):,} planetas)")
print(f"  Média: {sim_com['similaridade_terra'].mean():.1f}% | "
      f"Mediana: {sim_com['similaridade_terra'].median():.1f}% | "
      f"Máx: {sim_com['similaridade_terra'].max():.1f}%")
print("  Top 10:")
for _, r in sim_com.nlargest(10, 'similaridade_terra').iterrows():
    print(f"    {r['nome_planeta']:30} {r['similaridade_terra']:5.1f}%")

print(f"\nSEM remoção de outliers ({len(sim_sem):,} planetas)")
print(f"  Média: {sim_sem['similaridade_terra'].mean():.1f}% | "
      f"Mediana: {sim_sem['similaridade_terra'].median():.1f}% | "
      f"Máx: {sim_sem['similaridade_terra'].max():.1f}%")
print("  Top 10:")
for _, r in sim_sem.nlargest(10, 'similaridade_terra').iterrows():
    print(f"    {r['nome_planeta']:30} {r['similaridade_terra']:5.1f}%")

**Observação chave**: procure por TRAPPIST-1 e/f/g (candidatos conhecidos a Earth-analog) e por Proxima b. Se aparecem alto no ranking do cenário SEM outliers mas não no COM, é sinal de que o filtro IQR estava distorcendo a métrica de similaridade — porque no espaço z-score com scaler enviesado, distâncias pequenas viram grandes.

## 5. Visualização PCA lado a lado

Onde a Terra e o Sistema Solar caem no espaço PCA de cada pipeline.

In [ ]:
# DFsvm.xlsx não tem PC1/PC2 (foram removidas); uso DFLabelPropagation
df_com_pca = pd.read_excel(OUT_COM / "DFLabelPropagation.xlsx", index_col=0)
df_sem_pca = pd.read_excel(OUT_SEM / "DFLabelPropagation.xlsx", index_col=0)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, df_pipe, res_pipe, titulo in [
    (axes[0], df_com_pca, res_com, "COM remoção de outliers"),
    (axes[1], df_sem_pca, res_sem, "SEM remoção de outliers"),
]:
    # exoplanetas por cluster
    for c in sorted(df_pipe['cluster_hc'].unique()):
        mask = df_pipe['cluster_hc'] == c
        ax.scatter(df_pipe.loc[mask, 'PC1'], df_pipe.loc[mask, 'PC2'],
                   s=15, alpha=0.4, label=f'Cluster {c} (n={mask.sum()})')

    # projeta o Sistema Solar no MESMO espaço PCA
    corpos = diagnostics.CORPOS_REFERENCIA
    X_std = res_pipe['scaler'].transform(corpos[diagnostics.COLUNAS_NUMERICAS])
    X_pca = res_pipe['pca_std'].transform(X_std)

    for i, nome in enumerate(corpos['nome']):
        cor = 'red' if nome == 'Terra' else 'black'
        marker = '*' if nome == 'Terra' else 'x'
        tam = 350 if nome == 'Terra' else 80
        ax.scatter(X_pca[i, 0], X_pca[i, 1], c=cor, marker=marker, s=tam,
                   edgecolors='white', linewidths=1.5, zorder=10, label=None)
        ax.annotate(nome, (X_pca[i, 0], X_pca[i, 1]),
                    xytext=(6, 6), textcoords='offset points',
                    fontsize=9, color=cor, fontweight='bold', zorder=11)

    ax.set_title(titulo, fontsize=13, fontweight='bold')
    ax.set_xlabel(f"PC1 ({res_pipe['pca_std'].explained_variance_ratio_[0]:.1%} var.)")
    ax.set_ylabel(f"PC2 ({res_pipe['pca_std'].explained_variance_ratio_[1]:.1%} var.)")
    ax.legend(loc='best', fontsize=8)
    ax.grid(True, linestyle='--', alpha=0.3)

plt.suptitle("Onde caem os corpos do Sistema Solar no espaço PCA de cada pipeline",
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'outputs' / 'figs' / 'comparacao_outliers_pca.png',
            dpi=120, bbox_inches='tight')
plt.show()

## 6. Conclusão

Os dois cenários revelam o mesmo problema por lados opostos:

**COM remoção IQR (default do teu pipeline):**
- Dataset limpo, ~2739 planetas em distribuição "comportada"
- SVM converge, atinge alta acurácia interna
- **Mas a Terra fica em `|z|_max ≈ 9`** — fora da distribuição
- Terra é classificada como NT
- Similaridade máxima entre exoplanetas e Terra ≈ 79%

**SEM remoção:**
- Dataset completo, ~3773 planetas com toda a variabilidade
- Clustering degenera: uma anã marrom (VHS J1256b) domina uma dimensão inteira
- LP colapsa quase tudo em TT, SVM não consegue treinar
- **Mas a Terra fica em `|z|_max` bem menor** — perto da distribuição
- Similaridade da Terra faz sentido: TRAPPIST-1 e/d/h aparecem no topo

**O problema real**: o filtro IQR é uma ferramenta estatística agnóstica. Ele remove *tudo* que se afasta da mediana, sem distinguir entre "outlier ruim" (medição errada, brown dwarf) e "outlier interessante" (planeta tipo Terra num catálogo com viés de detecção pra hot jupiters).

**Próximo passo experimental**: substituir `rm_outliers` por um filtro físico + remoção só de outliers extremos (`|z| > 8` no dataset bruto, por exemplo), preservando a região Earth-like. Ou usar métrica robusta (Mahalanobis com covariância robusta, ou distância de Manhattan em vez de Euclidiana) que não seja tão sensível ao *scale* de cada dimensão.

Isso é uma discussão metodológica que pode virar um capítulo do teu relatório da IC — vai muito além de bug fix, é um resultado científico legítimo sobre o pipeline.

---

**Limpeza**: os arquivos temporários criados aqui (`_tmp_*`) podem ser removidos.

In [ ]:
for path in [OUT_COM / "_tmp_com.xlsx", OUT_COM / "_tmp_ranges.json",
             OUT_SEM / "_tmp_sem.xlsx", OUT_SEM / "_tmp_ranges.json"]:
    if path.exists():
        path.unlink()
        print(f"Removido: {path.name}")